In [11]:
language = 'pt'

# 1. Gravação de Áudio Com Python 🎤

In [13]:
from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

# Código JavaScript para gravar áudio do usuário usando a "MediaStream Recording API"
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
  # Executa o código JavaScript para gravar o áudio
  display(Javascript(RECORD))
  # Recebe o áudio gravado como resultado do JavaScript
  js_result = output.eval_js('record(%s)' % (sec * 1000))
   # Decodifica o áudio em base64
  audio = b64decode(js_result.split(',')[1])
  # Salva o áudio em um arquivo
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  # Retorna o caminho do arquivo de áudio (pasta padrão do Google Colab)
  return f'/content/{file_name}'

# Grava o áudio do usuário por um tempo determinado (padrão 5 segundos)
print('Ouvindo...\n')
record_file = record()

# Exibe o áudio gravado
display(Audio(record_file, autoplay=False))

Ouvindo...



<IPython.core.display.Javascript object>

# 2. Reconhecimento de Fala com Whisper (Groq) 🧠

In [14]:
!pip install groq gTTS

In [15]:
import os
os.environ["GROQ_API_KEY"] = "Sua_chave_aqui"

In [18]:
from groq import Groq

client = Groq()

with open(record_file, "rb") as file:
    transcription = client.audio.transcriptions.create(
        file=file,
        model="whisper-large-v3"
    )

texto = transcription.text
print("📝 Transcrição:", texto)

📝 Transcrição:  Me explique o conceito de inflação.


# 3. Integração com a API do GROQ 💬

In [21]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",  # ✅ modelo atualizado
    messages=[
        {
            "role": "system",
            "content": """Você é um especialista em finanças.
Explique de forma simples, com exemplos e linguagem acessível."""
        },
        {
            "role": "user",
            "content": texto
        }
    ],
    temperature=0.5
)

resposta = response.choices[0].message.content
print("🤖 Resposta:", resposta)

🤖 Resposta: A inflação é um conceito importante em economia e finanças. Vou explicar de forma simples e fácil de entender.

**O que é inflação?**

A inflação é uma situação em que o preço dos produtos e serviços aumenta ao longo do tempo. Isso significa que você precisa pagar mais dinheiro para comprar as mesmas coisas que antes.

**Exemplo:**

Imagine que você tem R$ 100 e pode comprar 10 pizzas com essa quantia. Se a inflação for de 10%, significa que os preços das pizzas aumentaram 10% e agora você precisa pagar R$ 110 para comprar as mesmas 10 pizzas. Se você tiver R$ 110, agora você pode comprar apenas 9,09 pizzas (calculado com base no preço antigo).

**Por que a inflação ocorre?**

A inflação pode ocorrer por vários motivos, como:

1. **Demanda e oferta**: Quando há uma alta demanda por produtos e serviços e uma oferta limitada, os preços tendem a aumentar.
2. **Crescimento econômico**: Quando a economia cresce rapidamente, os preços tendem a aumentar devido à demanda por mais p

# 4. Sintetizando a Resposta do ChatGPT Como Voz (gTTS) 🔊

In [ ]:
!pip install gTTS

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [22]:
from gtts import gTTS

tts = gTTS(text=resposta, lang='pt')

output_file = "resposta.mp3"
tts.save(output_file)

display(Audio(output_file, autoplay=True))